# Homework 3: Classification (ML Zoomcamp 2026)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
!wget -nc https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

--2026-10-12 21:54:32--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.02s   

2026-10-12 21:54:33 (17.4 MB/s) - ‘course_lead_scoring_2026.csv’ saved [278746/278746]



,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation
Check missing values, then fill categorical with `'NA'` and numerical with `0.0`.

In [3]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
target = 'converted'
categorical = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
numerical = [col for col in df.columns if col not in categorical + [target]]

df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

print('categorical:', categorical)
print('numerical:', numerical)
df.isnull().sum().sum()

categorical: ['lead_source', 'industry', 'employment_status', 'location']
numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


np.int64(0)

## Question 1: mode of `industry`

In [5]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [6]:
df['industry'].mode()[0]

'technology'

## Question 2: biggest correlation among the listed pairs

In [7]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [8]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(f'{a} / {b}: {corr.loc[a, b]:.4f}')

interaction_count / lead_score: 0.9157
number_of_courses_viewed / lead_score: 0.7572
number_of_courses_viewed / interaction_count: 0.7216
annual_income / interaction_count: 0.1228


## Split the data

In [9]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

df_train = df_train.drop(columns=target).reset_index(drop=True)
df_val = df_val.drop(columns=target).reset_index(drop=True)
df_test = df_test.drop(columns=target).reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

## Question 3: mutual information (training set only)

In [10]:
mi = {col: round(mutual_info_score(df_train[col], y_train), 2) for col in categorical}
pd.Series(mi).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

In [11]:
# unrounded, in case of ties after rounding
pd.Series({col: mutual_info_score(df_train[col], y_train) for col in categorical}).sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

## Question 4: logistic regression with one-hot encoding

In [12]:
def train_and_score(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

all_features = categorical + numerical
base_acc = train_and_score(all_features)
print(base_acc, round(base_acc, 2))

0.645 0.65


## Question 5: feature elimination

In [13]:
diffs = {}
for f in all_features:
    feats = [x for x in all_features if x != f]
    acc = train_and_score(feats)
    diffs[f] = base_acc - acc
pd.Series(diffs).sort_values()

annual_income              -0.079
industry                    0.000
location                    0.000
employment_status           0.000
lead_score                  0.001
number_of_courses_viewed    0.002
lead_source                 0.003
interaction_count           0.044
dtype: float64

In [14]:
{f: diffs[f] for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']}

{'lead_source': 0.0030000000000000027,
 'number_of_courses_viewed': 0.0020000000000000018,
 'interaction_count': 0.04400000000000004}

## Question 6: regularization

In [15]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_score(all_features, C=C)
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
